In [ ]:
label_folder = "../../input/lct_p3"
model_folder = "model_output/Llama-3-70B-Instruct_3_shot/output"

In [31]:
import os
import json
import re

label_folder = "../../input/lct_p3"
model_folder = "model_output/Llama-3-70B-Instruct_3_shot/output"

categories = ['Condition']
#categories = ['Condition', 'Observation', 'Drug']

def visit(node, category, entities):
    if isinstance(node, dict):
        if category in node:
            entities.extend(node[category])
        for value in node.values():
            visit(value, category, entities)
    elif isinstance(node, list):
        for item in node:
            visit(item, category, entities)

def calculate_metrics(label_data, model_data, metrics):
    for category in categories:
        label_entities = []
        visit(label_data, category, label_entities)
        model_entities = []
        visit(model_data, category, model_entities)
        print("Label data")
        print(label_data)
        print("Label entities")
        print(label_entities)
        print("Model entities")
        print(model_entities)
        

        true_positives = sum(entity in label_entities for entity in model_entities)
        false_positives = sum(entity not in label_entities for entity in model_entities)
        false_negatives = sum(entity not in model_entities for entity in label_entities)
        print("True positives:", true_positives)
        print("False positives:", false_positives)
        print("False negatives:", false_negatives)

        metrics[category]['true_positives'] += true_positives
        metrics[category]['false_positives'] += false_positives
        metrics[category]['false_negatives'] += false_negatives

def process_files(label_folder, model_folder):
    metrics = {category: {'true_positives': 0, 'false_positives': 0, 'false_negatives': 0} for category in categories}
    num_files = 0

    for filename in os.listdir(model_folder):
        if filename.endswith('.json'):
            model_path = os.path.join(model_folder, filename)
            match = re.search(r'(NCT\d+_exc)', filename)
            if match:
                label_filename = f"{match.group(1)}_p3.json"
                label_path = os.path.join(label_folder, label_filename)

                if os.path.exists(label_path):
                    try:
                        with open(label_path, 'r', encoding='utf-8') as label_file, open(model_path, 'r', encoding='utf-8') as model_file:
                            label_data = json.load(label_file)
                            model_data = json.load(model_file)
                            calculate_metrics(label_data, model_data, metrics)
                        num_files += 1
                    except json.JSONDecodeError as e:
                        print(f"Error decoding JSON file: {model_path}")
                        print(f"Error message: {str(e)}")

    if num_files > 0:
        for category in categories:
            true_positives = metrics[category]['true_positives']
            false_positives = metrics[category]['false_positives']
            false_negatives = metrics[category]['false_negatives']

            precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
            recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
            accuracy = true_positives / (true_positives + false_positives + false_negatives) if (true_positives + false_positives + false_negatives) > 0 else 0

            metrics[category]['precision'] = precision
            metrics[category]['recall'] = recall
            metrics[category]['f1'] = f1
            metrics[category]['accuracy'] = accuracy
    else:
        print("No matching files found.")

    return metrics

metrics = process_files(label_folder, model_folder)



Label data
{'AND': {'left': {'raw_text': '1. abnormal folate levels', 'Observation': ['folate levels']}, 'right': {'OR': {'left': {'raw_text': '2. age > 21'}, 'right': {'raw_text': 'or less than 2'}}}}}
Label entities
[]
Model entities
[]
True positives: 0
False positives: 0
False negatives: 0
Error decoding JSON file: model_output/Llama-3-70B-Instruct_3_shot/output\Llama-3-70B-Instruct_NCT03860025_exc_3_shot.json
Error message: Expecting value: line 1 column 1 (char 0)
Label data
{'AND': {'left': {'AND': {'left': {'AND': {'left': {'AND': {'left': {'raw_text': '1. Subject has received anti-CD38 monoclonal antibody treatment previously;', 'Drug': ['anti-CD38 monoclonal antibody']}, 'right': {'raw_text': '2. Subject has received CAR-T cell therapy previously;'}}}, 'right': {'OR': {'left': {'raw_text': '3. Subject has previously received allogenic stem cell transplant,'}, 'right': {'raw_text': 'or subject has received autologous stem cell transplant within 3 months before administration o

In [32]:
print("Metrics:")
for category, scores in metrics.items():
    print(f"{category}:")
    print(f"  Precision: {scores['precision']:.4f}")
    print(f"  Recall: {scores['recall']:.4f}")
    print(f"  F1-score: {scores['f1']:.4f}")
    print(f"  Accuracy: {scores['accuracy']:.4f}")

Metrics:
Condition:
  Precision: 0.4570
  Recall: 0.5755
  F1-score: 0.5095
  Accuracy: 0.3418
